# 🧪 Laboratorio de Rendimiento
## Implementando Álgebra Lineal: desde Cero hasta NumPy

En este notebook vamos a resolver los **4 ejercicios** propuestos, siguiendo siempre la misma lógica de trabajo:

$$\text{① Implementación manual (for + listas)} \;\longrightarrow\; \text{② NumPy array + operación manual} \;\longrightarrow\; \text{③ Función optimizada de NumPy}$$

Para cada ejercicio vamos a:

1. Repasar brevemente el concepto matemático.
2. Programar las tres versiones.
3. Verificar que las tres dan el mismo resultado (sanity check).
4. Medir tiempos de ejecución con `time.perf_counter()`.

Al final armamos la tabla comparativa completa y sacamos conclusiones.

> 💡 **Por qué importa esto:** entender *por qué* NumPy es más rápido (y no solo *que* es más rápido) es una de las bases para escribir código de ciencia de datos eficiente.


In [ ]:
import numpy as np
import time
import random

# Fijamos una semilla para que los resultados sean reproducibles
random.seed(42)
np.random.seed(42)

def medir_tiempo(func, *args, repeticiones=5, **kwargs):
    """
    Ejecuta 'func' varias veces y devuelve el tiempo promedio en segundos.
    Repetimos varias veces porque una sola corrida puede ser ruidosa
    (el sistema operativo, la cache, otros procesos, etc.).
    """
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)
        fin = time.perf_counter()
        tiempos.append(fin - inicio)
    return resultado, sum(tiempos) / len(tiempos)

# Acá vamos a ir guardando los resultados para la tabla final
resultados_benchmark = {}


---
# 🔹 Ejercicio 1 — Producto Cruz

## Repaso teórico

Para dos vectores en $\mathbb{R}^3$, $\vec{a} = (a_1, a_2, a_3)$ y $\vec{b} = (b_1, b_2, b_3)$, el producto cruz se define como:

$$\vec{a} \times \vec{b} = \begin{pmatrix} a_2 b_3 - a_3 b_2 \\ a_3 b_1 - a_1 b_3 \\ a_1 b_2 - a_2 b_1 \end{pmatrix}$$

Una forma fácil de recordarlo es como el "determinante simbólico":

$$\vec{a} \times \vec{b} = \begin{vmatrix} \hat{i} & \hat{j} & \hat{k} \\ a_1 & a_2 & a_3 \\ b_1 & b_2 & b_3 \end{vmatrix}$$

⚠️ **Dato importante:** el producto cruz "clásico" (el que da como resultado *otro vector*) **solo está definido en $\mathbb{R}^3$** (y, como caso especial poco usado, en $\mathbb{R}^7$). No existe una noción directa de "cruz de dos vectores" en, por ejemplo, $\mathbb{R}^5$.

Lo que sí generaliza es el siguiente problema: dados **$N-1$ vectores en $\mathbb{R}^N$**, existe un único vector (salvo signo) ortogonal a todos ellos, calculado mediante el **determinante generalizado por cofactores**:

$$\vec{v} = \begin{vmatrix} \hat{e}_1 & \hat{e}_2 & \cdots & \hat{e}_N \\ a^{(1)}_1 & a^{(1)}_2 & \cdots & a^{(1)}_N \\ \vdots & \vdots & & \vdots \\ a^{(N-1)}_1 & a^{(N-1)}_2 & \cdots & a^{(N-1)}_N \end{vmatrix}$$

Esta es la generalización que vamos a implementar en la parte ③ del ejercicio.


## ① Implementación manual (for + listas)

In [ ]:
def cross_manual_3d(a, b):
    """Producto cruz para vectores de 3 dimensiones, sin NumPy."""
    resultado = [
        a[1]*b[2] - a[2]*b[1],
        a[2]*b[0] - a[0]*b[2],
        a[0]*b[1] - a[1]*b[0]
    ]
    return resultado

# Prueba rápida
a = [1, 2, 3]
b = [4, 5, 6]
print("cross_manual_3d:", cross_manual_3d(a, b))


cross_manual_3d: [-3, 6, -3]


## ② Implementación con `numpy.array` (operación manual)

In [ ]:
def cross_array_3d(a, b):
    """
    Mismo cálculo, pero los datos viven en un np.array.
    OJO: seguimos accediendo elemento a elemento con índices,
    NO usamos ninguna función especializada de NumPy para el cálculo.
    """
    a = np.array(a)
    b = np.array(b)
    resultado = np.array([
        a[1]*b[2] - a[2]*b[1],
        a[2]*b[0] - a[0]*b[2],
        a[0]*b[1] - a[1]*b[0]
    ])
    return resultado

print("cross_array_3d:", cross_array_3d(a, b))


cross_array_3d: [-3  6 -3]


## ③ Implementación optimizada de NumPy

In [ ]:
def cross_numpy_3d(a, b):
    """Usa la función nativa de NumPy, vectorizada en C."""
    return np.cross(np.array(a), np.array(b))

print("cross_numpy_3d:", cross_numpy_3d(a, b))

# Sanity check: las tres implementaciones deben coincidir
assert list(cross_manual_3d(a, b)) == list(cross_array_3d(a, b)) == list(cross_numpy_3d(a, b))
print("✅ Las tres implementaciones coinciden")


cross_numpy_3d: [-3  6 -3]
✅ Las tres implementaciones coinciden


## Generalización a N dimensiones

Como vimos arriba, para generalizar necesitamos **N-1 vectores en $\mathbb{R}^N$**. La estrategia es construir una matriz $(N-1) \times N$ con esos vectores como filas, y calcular el "vector normal" expandiendo por cofactores a lo largo de la primera fila simbólica $(\hat{e}_1, \dots, \hat{e}_N)$.

Esto es equivalente a: para cada componente $k$ del resultado, calcular $(-1)^{k}$ veces el determinante del *menor* que se obtiene al eliminar la columna $k$.


In [ ]:
def cross_manual_nd(vectores):
    """
    Generalización manual (sin NumPy) del producto cruz a N dimensiones.
    'vectores' es una lista de N-1 vectores, cada uno con N componentes.
    Devuelve un vector de N componentes ortogonal a todos ellos.
    """
    n = len(vectores[0])
    assert len(vectores) == n - 1, "Se necesitan exactamente N-1 vectores en R^N"

    def determinante_manual(m):
        # Determinante recursivo por expansión de cofactores (sin NumPy)
        tam = len(m)
        if tam == 1:
            return m[0][0]
        if tam == 2:
            return m[0][0]*m[1][1] - m[0][1]*m[1][0]
        det = 0
        for col in range(tam):
            menor = [fila[:col] + fila[col+1:] for fila in m[1:]]
            signo = (-1) ** col
            det += signo * m[0][col] * determinante_manual(menor)
        return det

    resultado = []
    for k in range(n):
        # Menor: sacamos la columna k de la matriz de vectores
        menor = [fila[:k] + fila[k+1:] for fila in vectores]
        signo = (-1) ** k
        resultado.append(signo * determinante_manual(menor))
    return resultado


# Ejemplo en R^4: necesitamos 3 vectores
v1 = [1, 0, 0, 2]
v2 = [0, 1, 0, 3]
v3 = [0, 0, 1, 4]
print("cross_manual_nd (R^4):", cross_manual_nd([v1, v2, v3]))


cross_manual_nd (R^4): [2, 3, 4, -1]


In [ ]:
def cross_array_nd(vectores):
    """Misma idea, pero guardando todo en np.array (sin usar np.linalg.det)."""
    n = len(vectores[0])
    assert len(vectores) == n - 1

    def determinante_manual(m):
        tam = len(m)
        if tam == 1:
            return m[0, 0]
        if tam == 2:
            return m[0, 0]*m[1, 1] - m[0, 1]*m[1, 0]
        det = 0
        for col in range(tam):
            menor = np.delete(m[1:], col, axis=1)
            signo = (-1) ** col
            det += signo * m[0, col] * determinante_manual(menor)
        return det

    M = np.array(vectores)
    resultado = np.zeros(n)
    for k in range(n):
        menor = np.delete(M, k, axis=1)
        signo = (-1) ** k
        resultado[k] = signo * determinante_manual(menor)
    return resultado

print("cross_array_nd (R^4):", cross_array_nd([v1, v2, v3]))


cross_array_nd (R^4): [ 2.  3.  4. -1.]


In [ ]:
def cross_numpy_nd(vectores):
    """
    Versión optimizada: usamos np.linalg.det sobre cada menor.
    Esta es la forma 'idiomática' de resolverlo con NumPy.
    """
    n = len(vectores[0])
    assert len(vectores) == n - 1
    M = np.array(vectores, dtype=float)
    resultado = np.zeros(n)
    for k in range(n):
        menor = np.delete(M, k, axis=1)
        resultado[k] = ((-1) ** k) * np.linalg.det(menor)
    return resultado

print("cross_numpy_nd (R^4):", cross_numpy_nd([v1, v2, v3]))

# Sanity check en R^4
r1 = cross_manual_nd([v1, v2, v3])
r2 = list(cross_array_nd([v1, v2, v3]))
r3 = list(cross_numpy_nd([v1, v2, v3]))
print("✅ Coinciden:" , np.allclose(r1, r2) and np.allclose(r2, r3))


cross_numpy_nd (R^4): [ 2.  3.  4. -1.]
✅ Coinciden: True


## ⏱ Benchmark — Producto Cruz (caso 3D)

In [ ]:
a_big = [random.uniform(-10, 10) for _ in range(3)]
b_big = [random.uniform(-10, 10) for _ in range(3)]

_, t_manual = medir_tiempo(cross_manual_3d, a_big, b_big, repeticiones=10000)
_, t_array  = medir_tiempo(cross_array_3d, a_big, b_big, repeticiones=10000)
_, t_numpy  = medir_tiempo(cross_numpy_3d, a_big, b_big, repeticiones=10000)

resultados_benchmark["Producto Cruz (3D)"] = (t_manual, t_array, t_numpy)

print(f"For (manual):     {t_manual:.8f} s")
print(f"NumPy array:       {t_array:.8f} s")
print(f"np.cross:          {t_numpy:.8f} s")


For (manual):     0.00000088 s
NumPy array:       0.00000754 s
np.cross:          0.00005104 s


> 🔎 **Observación esperada:** en un vector tan chico como uno de 3 componentes, el *overhead* de crear objetos `np.array` puede hacer que la versión ② (o incluso ③) sea más lenta que el simple `for` con listas. NumPy brilla cuando el volumen de datos crece, no en vectores diminutos.

---
# 🔹 Ejercicio 2 — Producto Punto

## Repaso teórico

$$\vec{a} \cdot \vec{b} = \sum_{i=1}^{N} a_i \, b_i$$

A diferencia del producto cruz, el producto punto **sí generaliza directamente a cualquier N**, porque es simplemente una suma de productos componente a componente.


## ① Implementación manual (for + listas) — 3D y N-D en la misma función

In [ ]:
def dot_manual(a, b):
    """Producto punto manual. Sirve tanto para 3D como para N dimensiones."""
    assert len(a) == len(b), "Los vectores deben tener la misma dimensión"
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

a3, b3 = [1, 2, 3], [4, 5, 6]
print("dot_manual (3D):", dot_manual(a3, b3))

a_nd = [1, 2, 3, 4, 5]
b_nd = [5, 4, 3, 2, 1]
print("dot_manual (5D):", dot_manual(a_nd, b_nd))


## ② Implementación con `numpy.array` (operación manual)

In [ ]:
def dot_array(a, b):
    a = np.array(a)
    b = np.array(b)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

print("dot_array (3D):", dot_array(a3, b3))
print("dot_array (5D):", dot_array(a_nd, b_nd))


## ③ Implementación optimizada de NumPy

In [ ]:
v2= ...
v1= ...
resultado_calculculo_en_unpaso= ...

In [ ]:
def dot_numpy(a, b):
    return np.dot(np.array(a), np.array(b))

print("dot_numpy (3D):", dot_numpy(a3, b3))
print("dot_numpy (5D):", dot_numpy(a_nd, b_nd))

assert dot_manual(a3, b3) == dot_array(a3, b3) == dot_numpy(a3, b3)
assert dot_manual(a_nd, b_nd) == dot_array(a_nd, b_nd) == dot_numpy(a_nd, b_nd)
print("✅ Las tres implementaciones coinciden en 3D y en N-D")


## ⏱ Benchmark — Producto Punto (vector grande, N=100.000)

In [ ]:
N = 100_000
a_grande = [random.uniform(-1, 1) for _ in range(N)]
b_grande = [random.uniform(-1, 1) for _ in range(N)]

_, t_manual = medir_tiempo(dot_manual, a_grande, b_grande, repeticiones=20)
_, t_array  = medir_tiempo(dot_array, a_grande, b_grande, repeticiones=20)
_, t_numpy  = medir_tiempo(dot_numpy, a_grande, b_grande, repeticiones=20)

resultados_benchmark["Producto Punto (N=100.000)"] = (t_manual, t_array, t_numpy)

print(f"For (manual):  {t_manual:.6f} s")
print(f"NumPy array:    {t_array:.6f} s")
print(f"np.dot:         {t_numpy:.6f} s")
print(f"\nnp.dot fue ~{t_manual/t_numpy:.1f}x más rápido que el for manual")


> 🔎 **¿Qué esperar acá?** `np.dot` delega el cálculo a rutinas compiladas (BLAS), evitando el *bytecode overhead* del intérprete de Python en cada iteración — normalmente gana con margen a partir de vectores de miles de elementos. Sin embargo, el `for` manual sobre listas de Python también puede ser sorprendentemente competitivo en este caso puntual (suma + multiplicación escalar, sin creación de arrays intermedios), mientras que la versión ② (`numpy array` con `for` explícito) suele ser la **más lenta de las tres**: paga el costo de acceder a elementos de un array de NumPy uno por uno desde Python, sin aprovechar la vectorización. Este es un buen ejemplo de que "usar NumPy" no alcanza — hay que usar sus **funciones vectorizadas**, no solo sus arrays. Corré la celda y comparalo con tus propios números.

---
# 🔹 Ejercicio 3 — Producto Matricial (Matrices Cuadradas)

## Repaso teórico

Para $A, B \in \mathbb{R}^{M \times M}$, el producto $C = A \cdot B$ se define elemento a elemento como:

$$C_{ij} = \sum_{k=1}^{M} A_{ik} \, B_{kj}$$

Esto requiere **tres bucles anidados**: uno para las filas de $A$, uno para las columnas de $B$, y uno para la suma sobre $k$.


## ① Implementación manual (for anidados + listas) — caso 2×2

In [ ]:
def matmul_manual(A, B):
    """
    Producto matricial manual con listas de listas.
    Funciona para matrices cuadradas de cualquier tamaño M x M.
    """
    M = len(A)
    assert len(A[0]) == len(B) == M, "Dimensiones incompatibles"

    C = [[0 for _ in range(M)] for _ in range(M)]
    for i in range(M):
        for j in range(M):
            suma = 0
            for k in range(M):
                suma += A[i][k] * B[k][j]
            C[i][j] = suma
    return C

A2 = [[1, 2], [3, 4]]
B2 = [[5, 6], [7, 8]]
print("matmul_manual (2x2):")
for fila in matmul_manual(A2, B2):
    print(fila)


## ② Implementación con `numpy.array` (operación manual)

In [ ]:
def matmul_array(A, B):
    A = np.array(A)
    B = np.array(B)
    M = A.shape[0]
    C = np.zeros((M, M))
    for i in range(M):
        for j in range(M):
            suma = 0
            for k in range(M):
                suma += A[i, k] * B[k, j]
            C[i, j] = suma
    return C

print("matmul_array (2x2):\n", matmul_array(A2, B2))


## ③ Implementación optimizada de NumPy

In [ ]:
def matmul_numpy(A, B):
    return np.array(A) @ np.array(B)   # equivalente a np.matmul(A, B)

print("matmul_numpy (2x2):\n", matmul_numpy(A2, B2))

assert np.allclose(matmul_manual(A2, B2), matmul_array(A2, B2))
assert np.allclose(matmul_array(A2, B2), matmul_numpy(A2, B2))
print("✅ Las tres implementaciones coinciden")


## Generalización a M×M y benchmark

In [ ]:
def generar_matriz(M):
    return [[random.uniform(-5, 5) for _ in range(M)] for _ in range(M)]

M = 60  # con for anidados en Python puro ya empieza a notarse el costo O(M^3)
A_MM = generar_matriz(M)
B_MM = generar_matriz(M)

_, t_manual = medir_tiempo(matmul_manual, A_MM, B_MM, repeticiones=3)
_, t_array  = medir_tiempo(matmul_array, A_MM, B_MM, repeticiones=3)
_, t_numpy  = medir_tiempo(matmul_numpy, A_MM, B_MM, repeticiones=3)

resultados_benchmark[f"Matriz {M}x{M}"] = (t_manual, t_array, t_numpy)

print(f"For anidados (manual): {t_manual:.6f} s")
print(f"NumPy array (for):      {t_array:.6f} s")
print(f"A @ B:                  {t_numpy:.6f} s")
print(f"\nA @ B fue ~{t_manual/t_numpy:.1f}x más rápido que el for manual")


> 🔎 **Complejidad $O(M^3)$:** el producto matricial es el ejemplo clásico donde la diferencia entre Python puro y NumPy se vuelve dramática, porque el costo crece al cubo del tamaño. NumPy usa BLAS optimizado (multi-threaded, con *cache blocking*), algo imposible de igualar con `for` puro.

---
# 🔹 Ejercicio 4 — Producto Matricial (Matrices Rectangulares)

## Repaso teórico

Ahora $A \in \mathbb{R}^{m \times n}$ y $B \in \mathbb{R}^{n \times p}$ (la cantidad de **columnas de A** debe coincidir con la cantidad de **filas de B**). El resultado $C \in \mathbb{R}^{m \times p}$:

$$C_{ij} = \sum_{k=1}^{n} A_{ik} \, B_{kj} \qquad i=1,\dots,m \;\; j=1,\dots,p$$

Por ejemplo: $A$ de $3\times 2$ multiplicada por $B$ de $2\times 4$ da como resultado una matriz de $3\times 4$.


## ① Implementación manual (for anidados + listas) — caso 3×2

In [ ]:
def matmul_rect_manual(A, B):
    """
    Producto matricial manual para matrices rectangulares compatibles.
    A: m x n   B: n x p   ->   C: m x p
    """
    m = len(A)
    n = len(A[0])
    n_b = len(B)
    p = len(B[0])
    assert n == n_b, f"Dimensiones incompatibles: A es {m}x{n}, B es {n_b}x{p}"

    C = [[0 for _ in range(p)] for _ in range(m)]
    for i in range(m):
        for j in range(p):
            suma = 0
            for k in range(n):
                suma += A[i][k] * B[k][j]
            C[i][j] = suma
    return C

A32 = [[1, 2], [3, 4], [5, 6]]   # 3x2
B23 = [[7, 8, 9], [10, 11, 12]]  # 2x3
print("matmul_rect_manual (3x2 · 2x3 -> 3x3):")
for fila in matmul_rect_manual(A32, B23):
    print(fila)


## ② Implementación con `numpy.array` (operación manual)

In [ ]:
def matmul_rect_array(A, B):
    A = np.array(A)
    B = np.array(B)
    m, n = A.shape
    n_b, p = B.shape
    assert n == n_b, "Dimensiones incompatibles"

    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            suma = 0
            for k in range(n):
                suma += A[i, k] * B[k, j]
            C[i, j] = suma
    return C

print("matmul_rect_array (3x2 · 2x3):\n", matmul_rect_array(A32, B23))


## ③ Implementación optimizada de NumPy

In [ ]:
def matmul_rect_numpy(A, B):
    return np.array(A) @ np.array(B)

print("matmul_rect_numpy (3x2 · 2x3):\n", matmul_rect_numpy(A32, B23))

assert np.allclose(matmul_rect_manual(A32, B23), matmul_rect_array(A32, B23))
assert np.allclose(matmul_rect_array(A32, B23), matmul_rect_numpy(A32, B23))
print("✅ Las tres implementaciones coinciden")

# Sanity check extra: verificar que se detecta incompatibilidad de dimensiones
try:
    matmul_rect_manual([[1, 2, 3]], [[1, 2], [3, 4]])
except AssertionError as e:
    print("✅ Detecta correctamente dimensiones incompatibles:", e)


## Generalización a M×N y benchmark

In [ ]:
def generar_matriz_rect(filas, columnas):
    return [[random.uniform(-5, 5) for _ in range(columnas)] for _ in range(filas)]

m, n, p = 50, 70, 40   # A: 50x70   B: 70x40   ->   C: 50x40
A_MN = generar_matriz_rect(m, n)
B_NP = generar_matriz_rect(n, p)

_, t_manual = medir_tiempo(matmul_rect_manual, A_MN, B_NP, repeticiones=3)
_, t_array  = medir_tiempo(matmul_rect_array, A_MN, B_NP, repeticiones=3)
_, t_numpy  = medir_tiempo(matmul_rect_numpy, A_MN, B_NP, repeticiones=3)

resultados_benchmark[f"Matriz {m}x{n} · {n}x{p}"] = (t_manual, t_array, t_numpy)

print(f"For anidados (manual): {t_manual:.6f} s")
print(f"NumPy array (for):      {t_array:.6f} s")
print(f"A @ B:                  {t_numpy:.6f} s")
print(f"\nA @ B fue ~{t_manual/t_numpy:.1f}x más rápido que el for manual")


---
# 📊 Tabla comparativa final

Volcamos todos los tiempos que fuimos guardando en `resultados_benchmark`.


In [ ]:
print(f"{'Operación':<28}{'For (s)':>14}{'NumPy array (s)':>18}{'Función NumPy (s)':>20}{'Speedup':>12}")
print("-" * 92)
for operacion, (t_for, t_array, t_np) in resultados_benchmark.items():
    speedup = t_for / t_np if t_np > 0 else float('inf')
    print(f"{operacion:<28}{t_for:>14.8f}{t_array:>18.8f}{t_np:>20.8f}{speedup:>10.1f}x")


### 📝 Registro de resultados (completar según tu corrida)

| Operación                  | For | NumPy Array | Función NumPy | Speedup |
| --------------------------- | --: | -----------: | -------------: | ------: |
| Producto Punto (N=100.000)  |     |              |                |         |
| Producto Cruz (3D)          |     |              |                |         |
| Matriz 60×60                |     |              |                |         |
| Matriz 50×70 · 70×40         |     |              |                |         |

> Completá esta tabla copiando los valores que imprimió la celda anterior. Los números van a variar levemente según la máquina donde corras el notebook (Colab reasigna recursos entre sesiones).


---
# 🎯 Conclusiones

1. **En vectores/matrices chicas**, el `for` manual puede ser tan rápido —o incluso más rápido— que NumPy, porque crear arrays y llamar funciones de NumPy tiene un *costo fijo* (overhead) que no se amortiza con pocos datos.

2. **A medida que crece el tamaño del problema** (vectores de decenas de miles de elementos, matrices de decenas/cientos de filas y columnas), la ventaja de NumPy se vuelve exponencialmente más marcada, porque:
   - Evita el *bytecode overhead* del intérprete de Python en cada iteración.
   - Usa memoria contigua (arrays tipados) en lugar de listas de objetos Python dispersos en memoria.
   - Delega a rutinas BLAS/LAPACK compiladas y, muchas veces, *multi-threaded*.

3. **Claridad de código:** la versión ③ (NumPy optimizado) es sistemáticamente la más corta y legible. Esto no es casual: expresar la operación en términos de *álgebra lineal* (en vez de bucles explícitos) es más cercano a cómo pensamos el problema matemáticamente.

4. **El producto cruz es un buen ejemplo de que "generalizar" no siempre es trivial**: mientras el producto punto se extiende de forma directa a N dimensiones, el producto cruz clásico es intrínsecamente un fenómeno de $\mathbb{R}^3$, y su generalización requiere cambiar el problema (de "cruz entre 2 vectores" a "vector ortogonal a N-1 vectores").

> 💡 **Para explorar después:** repetí el benchmark de la Matriz M×M variando M (por ejemplo 10, 50, 100, 300) y graficá el tiempo vs. M para cada implementación con `matplotlib`. Vas a ver cómo la curva del `for` manual crece muchísimo más rápido (cúbicamente) que la de NumPy.


In [ ]:
from sympy import *
x=symbols('x')

In [ ]:
symbols

<function sympy.core.symbol.symbols(names, *, cls=<class 'sympy.core.symbol.Symbol'>, **args) -> 'Any'>

In [ ]:
x

x